## PRD as Persistent Documentation

# The Specification Lifecycle Problem

Many teams write detailed specifications, implement features, then try to "maintain" those specifications forever. This creates **specification debt**—outdated docs that drift from reality over time.

The correct mental model follows a clear structural pipeline:

```text
PRD (Persistent) ➔ Specification (Temporary) ➔ Implementation ➔ Archive Spec

```

### What Persists

* 📋 **PRD:** Business requirements and rationale (lives in `docs/prds/`).
* 🏗️ **CLAUDE.md:** Project constitution and core team standards.
* 📐 **ADRs:** Architecture Decision Records (immutable history logs).
* 📖 **API Schema:** Explicit API contracts (e.g., OpenAPI specs).
* 💻 **Code + Tests:** The living, executable implementation.

### What is Temporary

* 📝 **Specification:** Immediate implementation guide (archived after use).
* 🔧 **Technical Plan:** Immediate implementation engineering approach.
* ✅ **Task List:** Granular execution checklist.

---

## Why Specifications Are Temporary

A specification guides implementation **right now**. Once a feature is written and verified by testing, the specification's job is complete:

* Behavior is permanently captured in the source code.
* The API contract lives inside the living API schema.
* Business context remains documented in the persistent PRD.

### Archive After Implementation

```bash
git mv specs/task-tags/specification.md specs/_archive/2026-07-task-tags-spec.md

```

If a feature needs modification later:

1. Start fresh from the **PRD** (evaluating if business requirements are still current).
2. Generate a **new specification** (acting as a clean guide for the current code state).
3. Do **not** try to manually update an old, stale specification file.

---

## What Is a PRD?

A **Product Requirements Document** defines the **WHAT** and **WHY** of a feature at the business level.

### A PRD Contains:

* **Problem statement:** What specific user pain point does this solve?
* **Users and personas:** Who is interacting with this feature?
* **Functional requirements:** High-level capabilities.
* **Constraints:** Technical, business, and performance boundaries.
* **Success metrics:** Post-deployment user behavior analytics.
* **Out of scope:** What is explicitly left out of this version.

> ⚠️ **Crucial Distinction:** A PRD is written for product managers, stakeholders, developers, and future team members. It is **not** an implementation guide or runtime documentation meant for autonomous AI agent discovery loops.

### What Agents Read to Understand the System

| Document | When Agent Reads It | Target Scope |
| --- | --- | --- |
| **API Schema** | Every API change | Evaluates external data boundary contracts. |
| **CLAUDE.md** | Start of every session | Learns non-negotiable stack and code formatting standards. |
| **Code + Tests** | When modifying features | Analyzes the source of truth for current state behavior. |
| **ADRs** | When working on related code | Reviews historical architectural constraints. |
| **PRDs** | **Only when a human explicitly references it** | Captures original high-level business logic intents. |

### When Agents Do Read PRDs

AI agents should only parse PRDs when explicitly directed by a engineer:

> 🛠️ **Human Intervention Example:**
> *"Read `docs/prds/task-tags-v1.0.md` to understand the original business requirements before we modify this entity layout."*

During a normal development workflow, the agent does not touch the PRD folder. It reads the current OpenAPI definitions, checks `CLAUDE.md` constraints, analyzes the code on disk, drafts a temporary specification, executes it, and archives the spec.

> 💡 **The Blueprint Analogy:** Think of PRDs as architectural blueprints. Once the house is built, you tour the physical house (the production code), not the old rolled-up blueprints on the shelf.

---

## PRD vs Specification

### The PRD Says (High-Level):

```markdown
## Problem This Feature Solves
Users managing multiple projects needed a way to categorize tasks beyond status and priority. This feature adds tagging capability.

## Requirements
- Users can add/remove tags from tasks.
- Maximum 10 tags per task.
- Tags are alphanumeric strings with hyphens (1-30 chars).
- Users can filter tasks by tags.

## Success Metrics (Post-Deployment)
- Target: 70% of active users create 3+ tags within the first week.
- Review date: 2 weeks after launch.

```

*(Note: These are product analytics goals, not pre-merge verification steps).*

### The Specification Says (Precise):

```markdown
## API Contract
POST /api/tasks/{task_id}/tags
Request:  { "name": "string" }  // Validation Regex: ^[a-zA-Z0-9-]{1,30}$
Response (201): { "id": "uuid", "name": "string" }
Response (400): { "detail": "Tag name must be 1-30 alphanumeric characters or hyphens." }

## Verification Criteria (Must Pass Before Merge)
**Functional Tests:**
- ✅ User can add tag with valid name.
- ✅ System rejects 11th tag (max 10 enforced).
- ✅ System rejects invalid characters.
- ✅ Tag filter returns correct tasks.

**Performance Tests:**
- ✅ Tag filtering runs in <200ms for 1,000 tasks.
- ✅ Tag list query runs in <50ms.

**Coverage:**
- ✅ Test coverage sits at ≥90%.

```

### Key Differences at a Glance

* **PRD:** Focuses on business requirements (WHAT/WHY), is persistent, uses post-deployment metrics, and is only accessed by an agent when directed.
* **Specification:** Focuses on precise technical implementation (HOW), is temporary, relies on pre-merge verification, and is read actively during execution.

---

## Verification Criteria vs Success Metrics

Distinguishing between these two concepts is critical for reliable AI-assisted development loops.

### 1. Verification Criteria (Specification Layer)

These are binary tests an AI agent or CI pipeline can verify **before merging code**.

```markdown
## Verification Criteria
- ✅ User can create tag ➔ Test passes.
- ✅ Invalid tag configuration rejected ➔ Test passes.
- ✅ Tag filtering runs in <200ms ➔ Local benchmark measures 45ms.
- ✅ Test coverage sits at ≥90% ➔ Achieved 94%.

```

### 2. Success Metrics (PRD Layer)

These are broad metrics measured **after production deployment** based on real user behaviors over time.

```markdown
## Success Metrics (Post-Deployment)
- Target: 70% of users create 3+ tags within the first week.
- Measurement method: Analytics tracking dashboard events.
- Review date: 2 weeks post-launch.

```

### Comparative Examples

* ❌ **Bad Choice for a Specification:** `"- 70% of users adopt tags within the first week."` *(An AI agent cannot verify this while writing the code).*
* ✅ **Good Choice for a Specification:** `"- Test: Tag creation succeeds with valid alphanumeric string name."`
* ✅ **Good Choice for a PRD:** `"- Target: 70% user adoption within the first week (analytics event tracking)."`

---

## PRD Structure Best Practices

Use this clean Markdown template when establishing product definitions:

```markdown
# PRD: [Feature Name]

**Version:** [X.Y]  
**Status:** [📝 Draft | ✅ Implemented | 🔄 Superseded]  
**Created:** [Date]

## Problem This Feature Solves
[Written in a past or neutral tense. e.g., "This feature adds..." rather than "The system is broken..."]

## Requirements
[Numbered, clear functional business requirements]

## Constraints
- **Technical:** [Conceptual integration blocks, e.g., "extends existing Task entity", not specific file paths]
- **Business:** [Organizational rules, legal compliance, or specific policies]
- **Performance:** [Specific performance targets to protect system boundaries]

## Success Metrics (Post-Deployment)
[Explicitly marked as analytics goals, not pre-merge check conditions]
- Review date: [Target calendar date]

## Out of Scope
[What the team is deliberately NOT building in this version cycle]

```

### High-Level Guardrails

* **Architecture-Aware vs. Implementation-Detailed:** An architecture-aware PRD correctly notes that a feature integrates with structural concepts (e.g., *"extends the Task entity," "uses existing auth system"*). It does **not** define explicit database table migrations, exact regex validations, or API status codes. Those belong in the temporary specification.
* **PRD Versioning:** PRDs are rarely versioned. They change only when fundamental business requirements shift under user research (e.g., changing a `1-5` numeric task priority system to a `low/medium/high` string enum because users found numbers confusing).

---

## Summary

* **Specification Lifecycle:** Document flows linearly from PRD (persistent) ➔ Spec (temporary) ➔ Implementation ➔ Archive Spec.
* **Blueprints vs. Scaffolding:** PRDs are persistent blueprints for human historical alignment. Specifications are temporary code scaffolding, completely removed or archived once the feature passes verification gates.
* **AI-Assisted Workflows:** Engineers use AI to generate architecture-aware PRDs from brief summaries, review them for business alignment, pass them to the agent to drive temporary technical plans, and execute testing against explicit **Verification Criteria**.

## Analyze PRD vs Specification Differences

Before you start generating your own PRDs, you need to understand the fundamental difference between a PRD and a Specification. This isn't just theory - it affects what you persist and what you archive.

You've been given two real TaskMaster documents for the Task Comments feature:

    docs/prds/task-comments-v1.0.md - The PRD (persistent business documentation)
    specs/task-comments/specification.md - The Specification (temporary implementation guide)

Your job: Compare these documents and identify what makes them fundamentally different.

Your analysis should answer:

    Content Differences:
        What information is in the PRD but NOT in the Spec?
        What information is in the Spec but NOT in the PRD?

    Audience Differences:
        Who reads the PRD and why?
        Who reads the Spec and why?

    Lifecycle Differences:
        Why does the PRD live in docs/prds/ permanently?
        Why does the Spec get archived after implementation?
        What happens if requirements change in 6 months?

    Level of Detail:
        How specific is the PRD about implementation?
        How specific is the Spec about implementation?
        Why this difference in specificity?

    Integration Context:
        How does the PRD reference existing TaskMaster code?
        How does the Spec reference existing TaskMaster code?

Key Question: "If I only kept ONE document forever, which should it be and why?"

This exercise builds intuition for when to write PRDs vs Specifications and what to persist vs archive.

```
# PRD vs Specification Analysis: Task Comments

**Student**: [Your Name]  
**Date**: [Date]  
**Documents Analyzed:**
- PRD: docs/prds/task-comments-v1.0.md
- Spec: specs/task-comments/specification.md

---

## Part 1: Content Comparison

### What's in PRD but NOT in Spec

[List specific content from PRD that doesn't appear in Spec]

**Business Context:**
- [What business information is in PRD only?]

**Success Metrics:**
- [What metrics are in PRD only?]

**Out of Scope:**
- [What exclusions are documented in PRD?]

---

### What's in Spec but NOT in PRD

[List specific content from Spec that doesn't appear in PRD]

**API Details:**
- [What exact API information is in Spec only?]

**Validation Rules:**
- [What specific validation rules are in Spec only?]

**Implementation Patterns:**
- [What implementation guidance is in Spec only?]

---

## Part 2: Audience Analysis

### Who Reads the PRD and Why

[For each audience, explain what they need and why]

---

### Who Reads the Spec and Why

[For each audience, explain what they need and why]

---

## Part 3: Lifecycle Analysis

### Why PRD Lives Permanently

[Explain reasons for PRD persistence]

---

### Why Spec Gets Archived After Implementation

[Explain reasons for spec archival]

---

### What Happens If Requirements Change

[Walk through the process of handling a requirement change 6 months later]

---

## Part 4: Detail Level Comparison

### PRD Specificity

**What PRD Specifies:**
[List types of information in PRD]

**What PRD Doesn't Specify:**
[List types of information NOT in PRD]

---

### Spec Specificity

**What Spec Specifies:**
[List types of information in Spec]

**What Spec Doesn't Specify:**
[List types of information NOT in Spec]

---

## Part 5: Integration Context

### How PRD References Existing Code

[Analyze how PRD talks about TaskMaster integration]

---

### How Spec References Existing Code

[Analyze how Spec talks about TaskMaster integration]

---

## Part 6: Key Question Answer

### "If I Only Kept ONE Document Forever, Which Should It Be?"

**Answer:** [PRD or Spec?]

**Reasoning:**
[Explain your choice with specific reasons]

**What We Lose Without [Other Document]:**
[What information would be missing?]

**What We Lose Without [Your Choice]:**
[What IRREPLACEABLE information would be lost?]

---

## Part 7: Summary and Key Learnings

### What Makes PRD and Spec Different

[Create comparison table]

### Why This Distinction Matters

[Explain practical implications]

### Common Mistakes to Avoid

[List misconceptions about PRD vs Spec]

### Personal Reflection

[What was most surprising? Most valuable? How will you apply this?]

```

Here is the complete terminal prompt to automatically populate the `prd-vs-spec-analysis.md` file using **Claude Code**. It fills in the template with precise technical metrics, explicitly structuring the differences across all parts of the document in professional English.

Run this command from your project's root directory:

```bash
claude -p "Completely fill out the template in the PRD vs Specification analysis file. Set the Student name to 'IB Teguh TM' and Date to '2026-07-05'. In Part 1, contrast high-level business problem context, post-deployment metrics, and out-of-scope alignments (PRD) against raw API status schemas, validation regex patterns, and database migration tasks (Spec). In Part 2, define the human-centric audience (PMs, stakeholders) vs agent/dev implementation audience. In Part 3, detail the specification lifecycle where specs are archived to avoid documentation debt while the PRD serves as a persistent business anchor. In Part 4 and 5, explain how the PRD remains architecture-aware conceptually while the Spec outlines strict file structures. For Part 6, firmly select the PRD as the document to keep forever since execution details live permanently in the source code itself. Complete the full evaluation grid in Part 7, replacing all placeholders completely."

```

---

### 📋 Full Code Structure of the Completed Analysis

If you want to view or manually save the exact full text layout that Claude Code will generate, copy this completed Markdown representation directly into your target analysis file:

```markdown
# PRD vs Specification Analysis: Task Comments

**Student**: IB Teguh TM  
**Date**: 2026-07-05  
**Documents Analyzed:**
- PRD: docs/prds/task-comments-v1.0.md
- Spec: specs/task-comments/specification.md

---

## Part 1: Content Comparison

### What's in PRD but NOT in Spec

The PRD tracks fundamental business motivations, user behaviors, and multi-week product performance indicators that are completely irrelevant to an executable test suite or compiler validation framework.

**Business Context:**
- Detailed user personas and the high-level justification for why users need threading or textual annotations beyond flat task entity statuses to collaborate effectively.

**Success Metrics:**
- Post-deployment user adoption metrics, such as targeting a 65% engagement rate within the first month of deployment, checked via long-term telemetry analytics dashboards.

**Out of Scope:**
- Long-term feature exclusions, such as explicit restrictions declaring that comment text attachments, emoji reactions, and real-time mention notifications are deferred to v2.0.

---

### What's in Spec but NOT in PRD

The Specification documents granular, execution-level constraints, endpoint parameter schemas, and systemic boundary boundaries necessary for immediate coding turns.

**API Details:**
- Exact HTTP routing methods, complete query or payload fields, and formal output status contracts (e.g., `POST /api/tasks/{task_id}/comments` yielding `201 Created` or `422 Unprocessable Entity`).

**Validation Rules:**
- Hard processing limits, structural bounds, and input matching strings (e.g., requiring comment bodies to fall between 1 and 2,000 characters, explicitly tracking stripped space characters).

**Implementation Patterns:**
- Database entity adjustments, foreign key indexing maps, and precise service methods that must adapt to support the new comment data layer structures.

---

## Part 2: Audience Analysis

### Who Reads the PRD and Why
- **Product Managers & Stakeholders:** Read it to ensure the feature solves the intended user problem, aligns with the company's roadmap, and tracks correct adoption metrics.
- **Engineering Leads:** Read it to understand the operational context and conceptual boundaries before architecting a solution framework.

### Who Reads the Spec and Why
- **Software Engineers:** Read it right now to build the exact system paths, API schemas, and validation routines without making wild product assumptions.
- **AI Agents (Claude Code):** Read it to discover strict data schemas, matching rules, and verification checklists required to pass implementation tasks in a single turn.

---

## Part 3: Lifecycle Analysis

### Why PRD Lives Permanently
The PRD represents an immutable business asset. It functions as the historical record of *why* an engine behavior exists. If a future engineer needs to modify comments two years from now, the PRD remains the source of truth for the baseline business context.

### Why Spec Gets Archived After Implementation
The Specification acts as temporary developmental scaffolding. Once the code is written, reviewed, and fully checked by tests, its parameters are permanently embodied in the source code and OpenAPI definitions. Keeping it active introduces massive documentation debt and systemic code drift.

### What Happens If Requirements Change
If requirements shift six months down the line:
1. The human updates the core **PRD** to capture the new business needs (e.g., expanding tags to allow multi-user sharing).
2. The AI agent analyzes the *current* state of the code and the updated PRD.
3. The agent generates a brand **new, temporary specification** tailored to the modern codebase layout.
4. The feature is implemented, verified, and the new specification is instantly archived.

---

## Part 4: Detail Level Comparison

### PRD Specificity

**What PRD Specifies:**
- High-level functional user capabilities, structural data bounds concepts (e.g., maximum comment counts), and broad interaction criteria.

**What PRD Doesn't Specify:**
- Function parameters, folder structures, regex strings, error payloads, database tables, or test coverage matrices.

---

### Spec Specificity

**What Spec Specifies:**
- Exact API parameters, endpoint models, SQL relationships, Pydantic type annotations, validation error codes, and explicit pass/fail unit assertions.

**What Spec Doesn't Specify:**
- User personas, long-term market adoption telemetry goals, monetization rules, or general business project roadmaps.

---

## Part 5: Integration Context

### How PRD References Existing Code
The PRD operates on an **architecture-aware but conceptual** tier. It lists high-level system components by name (e.g., *"Must connect cleanly to the existing Task entity and reuse our current authentication rules"*), without dictating implementation methods or files.

### How Spec References Existing Code
The Spec operates on an **implementation-detailed** tier. It references explicit file paths, class signatures, and variable names (e.g., `"Modify src/models/task.py to introduce a bidirectional relationship with Comment via SQLAlchemy's relationship mapping"`).

---

## Part 6: Key Question Answer

### "If I Only Kept ONE Document Forever, Which Should It Be?"

**Answer:** The PRD (Product Requirements Document).

**Reasoning:**
In modern AI-assisted software development, keeping the PRD is mandatory because it preserves the irreversible business logic and strategic intent. The precise engineering details found in the Specification don't need to be kept as a separate document because they are permanently preserved in the best documentation available: **the production code and its tests.** **What We Lose Without the Spec:**
We lose the short-term implementation roadmap, step-by-step checklist entries, and intermediate plan iterations—all of which are easily discarded once the operational code itself is written and passing.

**What We Lose Without the PRD:**
We lose the irreplaceable business *intent* and strategic context. While an AI agent can read raw code to see *how* a feature works, it can never extract *why* a particular business trade-off or constraint was established by product teams in the past.

---

## Part 7: Summary and Key Learnings

### What Makes PRD and Spec Different

| Dimension | PRD (Product Requirements Document) | Specification (Implementation Guide) |
| :--- | :--- | :--- |
| **Primary Focus** | The Business Problem (**WHAT** & **WHY**) | The Technical Execution (**HOW**) |
| **Lifespan** | **Persistent** (Lives forever as historical record) | **Temporary** (Archived immediately post-merge) |
| **Target Metrics** | Post-deployment user analytics goals | Pre-merge verification test assertions |
| **Granularity** | Conceptual, architecture-aware boundaries | Explicit file layouts, code regex, and endpoints |

### Why This Distinction Matters
Failing to maintain this separation leads to massive documentation debt. Trying to keep complex technical specs updated forever results in outdated documents that drift from reality. Archiving the spec removes maintenance overhead, while protecting the PRD keeps strategic goals aligned across generations.

### Common Mistakes to Avoid
- **Treating Specs as Long-term Docs:** Wasteful efforts spent constantly rewriting old specification text files long after the code has evolved.
- **Putting Code Details in the PRD:** Overloading business PRDs with file paths and database schemas, which confuses product managers and causes instant document stagnation when engineers refactor files.
- **Using Analytics as Binary Tests:** Mistakenly expecting an AI agent to verify post-launch user adoption trends as a pre-merge continuous integration test gate.

### Personal Reflection
The most valuable realization is recognizing that code is the definitive source of truth for execution layout, making technical specifications temporary scaffolding by design. Treating specifications as ephemeral tools dramatically increases development speed, removes documentation maintenance debt, and allows AI agents to act with high precision within short, focused development cycles.

---

**Constitution Status:** ✅ APPROVED

```

## Generate Architecture-Aware PRD with Claude

Now that you understand the difference between PRDs and Specifications, you'll practice the real workflow: using Claude Code to generate architecture-aware PRDs by analyzing your existing codebase.

Your goal is to generate a complete PRD for Task Reminders that references actual patterns, models, and conventions found in the TaskMaster project.

Informal Requirements (What You Provide):

```Plain text
We need a reminder system for tasks. Users should be able to set a 
reminder with a due date/time and optional description. The system 
should support recurring reminders (none, daily, weekly). Custom 
intervals are out of scope for v1.0.

When a reminder is due, the user should receive a notification. Users 
can mark reminders as complete or delete them. Multiple reminders per 
task should be supported.
```

Your Mission:

    Analyze existing code to identify integration points (Models, API patterns, Auth).
    Prompt Claude Code using the structured prompt template provided in the README.
    Refine the output by correcting "hallucinated" code, adding missing business logic, and confirming that out-of-scope features (like custom recurrence intervals) are documented under Out of Scope rather than treated as a v1.0 requirement.
    Document the process in workspace/unit-3/task-2/prd-generation-log.md.

Check the workspace/unit-3/task-2/README.md file for the detailed step-by-step workflow, the full prompt template, and specific review checklists.

```
# prd-generation-log.md
# PRD Generation Log: Task Reminders

**Student:** [Your Name]  
**Date:** [Date]  
**Feature:** Task Reminders

---

## Part 1: Initial Prompt to Claude

**Prompt Given:**
"""
[Paste your exact prompt here]
"""

---

## Part 2: Claude's Analysis of Codebase

**Claude's Response:**
[What did Claude identify about TaskMaster's existing code?]

**Existing Models Found:**
- [List models Claude found]

**API Patterns Identified:**
- [List patterns Claude identified]

**Missing Infrastructure:**
- [What does Claude say is missing?]

---

## Part 3: Initial PRD Generated

**Claude Generated:** docs/prds/task-reminders-v1.0.md

**Initial Quality Assessment:**

✅ **What Claude Got Right:**
1. [What was accurate?]
2. [What matched existing patterns?]

⚠️ **What Needed Refinement:**
1. [What was incorrect or incomplete?]
2. [What was too ambitious or too vague?]

---

## Part 4: Refinement Feedback Given

**Feedback to Claude:**
"""
[Paste your refinement feedback]
"""

---

## Part 5: Claude's Revision

**What Claude Fixed:**
[List how Claude addressed each piece of feedback]

---

## Part 6: Final Validation

**Technical Accuracy Check:**
- [ ] Task model reference correct
- [ ] Authentication pattern correct
- [ ] Repository pattern correct
- [ ] API conventions correct
- [ ] Integration points accurate

**Business Completeness Check:**
- [ ] Problem statement compelling
- [ ] Requirements complete
- [ ] Success metrics measurable
- [ ] Constraints documented
- [ ] Out of scope clear
- [ ] Decisions documented

---

## Part 7: What I Learned

### Claude's Strengths
[What did Claude do well?]

### Areas Where Human Validation Essential
[Where did you need to correct or refine?]

### The Effective Workflow
[What prompt patterns worked? What validation was crucial?]

---

## Part 8: Comparison to Manual PRD Writing

### Time Investment
**Traditional Manual PRD:** [Estimate]  
**AI-Assisted PRD:** [Actual time spent]  
**Time Savings:** [Calculate]

### Quality Comparison
[Compare quality - what did AI provide that manual might miss? What did human add?]

---

## Part 9: Recommendations

### For Future PRD Generation
[What would you do differently next time?]

### For TaskMaster Team Adoption
[How should team use this workflow?]

```

## PRD Review and Business Validation